# Internal IT Ticketing: Survey (Assessment) Pipeline (PySpark / Microsoft Fabric)

Companion to the [incident pipeline](01_incident_pipeline.ipynb). Post-ticket satisfaction surveys come from ServiceNow's assessment tables (`asmt_*`) and move through the same bronze → silver → gold layers.

| Layer | Notebook section | Output tables |
|---|---|---|
| **Bronze** | raw ServiceNow extracts (input) | `asmt_instance_bronze`, `asmt_questions_bronze`, `asmt_answers_bronze` |
| **Silver** | Part 1: Bronze → Silver | `asmt_instance_silver`, `asmt_question_silver`, `asmt_answers_silver` |
| **Gold** | Part 2: Silver → Gold | `asmt_instance_gold`, `asmt_question_gold`, `asmt_answers_gold` |

Sample CSVs of each layer are in the repo's `data/` folder.


# Part 1: Bronze → Silver
Keep survey instances tied to incidents, drop unused columns, and cast numeric fields.


## Imports

In [ ]:
import pyspark.sql.functions as sf
from pyspark.sql import types as T
from pyspark.sql.functions import col, datediff, when, concat, lit, round as spark_round
from pyspark.sql import Window

## Load Silver Survey Data

In [ ]:
DB = "internal_it_100b_lkh"

#Pulls necessary columns
instance_frame = spark.sql(f"SELECT * FROM {DB}.asmt_instance_bronze")
question_frame = spark.sql(f"SELECT * FROM {DB}.asmt_questions_bronze")
answers_frame  = spark.sql(f"SELECT * FROM {DB}.asmt_answers_bronze")


## Instance Frame

In [ ]:
def clean_data(instance_frame):
    # Drop columns
    instance_frame = instance_frame.drop('related_table_4', 'channel', 'related_table_1', 'preview', 'related_table_2', 'signature_result', 'related_table_3', 'sys_domain_path', 'sys_domain', 'related_id_2', 'related_id_3', 'related_id_4', 'signature', 'notification_flow_context_id')
    # Filter rows based on column: 'trigger_table'
    instance_frame = instance_frame.filter(instance_frame['trigger_table'] == "incident")
    return instance_frame
instance_frame = clean_data(instance_frame)

## Question Frame

In [ ]:
def clean_data(question_frame):
    # Drop columns
    question_frame = question_frame.drop('sys_domain_path', 'condition', 'correct_answer_template', 'default_value_field', 'value_parameters', 'sys_policy', 'sys_class_name', 'hide_label', 'description', 'details', 'maximum_normalization_input', 'source_field', 'displayed_when_template', 'plugin', 'reference_table', 'custom_metric')
    # Drop columns: 'allow_not_applicable', 'read_only' and 20 other columns
    question_frame = question_frame.drop('allow_not_applicable', 'read_only', 'displayed_when', 'randomize_answers', 'auto_gen', 'active', 'duration', 'sys_package', 'correct_answer_checkbox', 'script', 'context', 'sys_updated_on', 'sys_created_on', 'displayed_when_checkbox', 'allow_add_info', 'max_weight', 'template', 'correct_answer', 'displayed_when_yesno', 'cond_question', 'field_validation', 'correct_answer_choice')
    

    # Change column type to float32 for columns: 'weight', 'min'
    question_frame = question_frame.withColumn('weight', question_frame['weight'].cast(T.FloatType()))
    question_frame = question_frame.withColumn('min', question_frame['min'].cast(T.FloatType()))
    return question_frame
question_frame = clean_data(question_frame)

## Answer Frame

In [ ]:
def clean_data(answers_frame):
    # Drop columns: 'sys_domain_path', 'duration_value' and 6 other columns
    answers_frame = answers_frame.drop('sys_domain_path', 'duration_value', 'is_default', 'nps_value', 'reference_table', 'reference_value', 'add_info', 'passed')
    # Drop columns: 'sys_mod_count', 'template_definition', 'sys_domain'
    answers_frame = answers_frame.drop('sys_mod_count', 'template_definition', 'sys_domain')

    # Change column type to float32 for columns: 'normalized_value', 'scaled_value' and 2 other columns
    answers_frame = answers_frame.withColumn('normalized_value', answers_frame['normalized_value'].cast(T.FloatType()))
    answers_frame = answers_frame.withColumn('scaled_value', answers_frame['scaled_value'].cast(T.FloatType()))
    answers_frame = answers_frame.withColumn('actual_value', answers_frame['actual_value'].cast(T.FloatType()))
    answers_frame = answers_frame.withColumn('weighted_value', answers_frame['weighted_value'].cast(T.FloatType()))
    return answers_frame
answers_frame = clean_data(answers_frame)

## Saving Tables

In [ ]:
#Saving Tables
silver_frames = {
    "asmt_instance_silver": instance_frame,
    "asmt_question_silver": question_frame,
    "asmt_answers_silver":  answers_frame,
}

SILVER_DB = "internal_it_200s_lkh"

for table_name, df in silver_frames.items():
    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{SILVER_DB}.{table_name}")

# Part 2: Silver → Gold
Keep the incident satisfaction survey only, derive question keys and one column per question, and remove test responses.


## Imports

In [ ]:
import pyspark.sql.functions as sf
from pyspark.sql import types as T
from pyspark.sql.functions import col, datediff, when, concat, lit, substring, round as spark_round
from pyspark.sql import Window

## Load Gold Survey Data

In [ ]:
DB = "internal_it_200s_lkh"

#--------------------Pulls necessary columns
instance_frame = spark.sql(f"SELECT * FROM {DB}.asmt_instance_silver")
question_frame = spark.sql(f"SELECT * FROM {DB}.asmt_question_silver")
answers_frame  = spark.sql(f"SELECT * FROM {DB}.asmt_answers_silver")

## Instance Frames

In [ ]:
def clean_data_instance(instance_frame):
    instance_frame = instance_frame.withColumn('sys_created_date', col('sys_created_on'))
    instance_frame = instance_frame.withColumn('sys_created_date', substring('sys_created_date', 1, 10))
    return instance_frame
instance_frame = clean_data_instance(instance_frame)

## Question Frame

In [ ]:
def clean_data_question(question_frame):
    # Sort by column: 'order' (ascending)
    question_frame = question_frame.sort(question_frame['order'].asc())

    # Filter rows based on column: 'metric_type'
    question_frame = question_frame.filter(question_frame['metric_type'].contains("79876d702b933a505f3cf6d6f291bf0a"))
    return question_frame
question_frame = clean_data_question(question_frame)

## Answer Frame

The answers table stores one row per question per survey. `assign_to_order` ends in a 3-digit question code (100 = satisfaction, 200 = resolved, 400 = repeat issue). The cell below extracts that code, builds a per-question key, and splits the answer into one column per question.


In [ ]:
def clean_data_answers(answers_frame):
    # Rename column 'string_value' to 'value_string'
    answers_frame = answers_frame.withColumnRenamed('string_value', 'value_string')

    # Derive question code and per-question key from assign_to_order
    answers_frame = answers_frame.orderBy(['assessment_group', 'assign_to_order'])
    answers_frame = answers_frame.withColumn('question_id',substring(col('assign_to_order'), -3, 3).cast('int'))
    answers_frame = answers_frame.withColumn('question_instance_id',concat(col('assessment_group'),lit('-'),substring(col('assign_to_order'), -3, 3)))

    # Filter rows
    answers_frame = answers_frame.filter(answers_frame['source_id'].contains("79876d702b933a505f3cf6d6f29"))
    answers_frame = answers_frame.filter(answers_frame['source_table'] == "asmt_metric_type")
    answers_frame = answers_frame.withColumnRenamed('string_value', 'value_string')

    #Derived Columns based on the value_string
    answers_frame = answers_frame.withColumn('value_satisfaction',when(col('question_id') == 100, col('value_string')).otherwise(None))
    answers_frame = answers_frame.withColumn('value_resolved',when(col('question_id') == 200, col('value_string')).otherwise(None))
    answers_frame = answers_frame.withColumn('value_repeat_issue',when(col('question_id') == 400, col('value_string')).otherwise(None))
    
    #Find all assessment_groups where any row has "Test" in string_value
    test_groups = (
    answers_frame.filter(col("value_string") == "Test")  # Exact match only — excludes "Testing", "Tested", etc.
    .select("assessment_group")
    .distinct()
    )
    #Remove ALL rows that belong to those assessment_groups
    answers_frame = answers_frame.join(test_groups, on="assessment_group", how="left_anti")

    return answers_frame
answers_frame = clean_data_answers(answers_frame)

## Saving Tables

In [ ]:
#------------------Saving Tables
gold_frames = {
    "asmt_instance_gold": instance_frame,
    "asmt_question_gold": question_frame,
    "asmt_answers_gold":  answers_frame,
}

GOLD_DB = "internal_it_300g_lkh"

for table_name, df in gold_frames.items():
    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{GOLD_DB}.{table_name}")